In [3]:
import ast

import pandas as pd
from utils.load import load_json_or_jsonl
from pathlib import Path

INPUT  = Path("../../data/dataset/applets/applets_synt_new_desc.jsonl")
OUT = Path("../../data/dataset/applets/applets_synt_new_filter.jsonl")
TRIGGER = Path("../../data/ifttt_catalog/triggers.json")
ACTION  =Path("../../data/ifttt_catalog/actions.json")
applets = load_json_or_jsonl(INPUT)
triggers = load_json_or_jsonl(TRIGGER)
actions  = load_json_or_jsonl(ACTION)
trigger_index = {t["api_endpoint_slug"]: t for t in triggers}
action_index = {a["api_endpoint_slug"]: a for a in actions}
df_orig = pd.DataFrame(applets)

In [4]:
from llm_utility.requests.parallel.response import process_rows_F, MAX_WORKERS

res = process_rows_F(applets,trigger_index,action_index, max_workers=MAX_WORKERS)
df_res = pd.DataFrame(res)
df_res

[F] 5999 error:False

In [6]:
df_res = pd.DataFrame(res)
df_plus=df_orig.merge(df_res,on='row_index', how='inner')
df_res=df_plus[df_plus.require_filter_code]

import math


# --- funzione per pulire NaN ---
def replace_nan_with_none(obj):
    if isinstance(obj, float) and math.isnan(obj):
        return None
    if isinstance(obj, dict):
        return {k: replace_nan_with_none(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [replace_nan_with_none(x) for x in obj]
    return obj

df_res.to_json(OUT,orient='records',lines=True,force_ascii=False)